In [1]:
import os
import subprocess
import shutil
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import multiprocessing as mp
from functools import partial
import time
import glob
import traceback
import json
import sys
import itertools

In [2]:
original_dir  = os.getcwd()
run_directory = original_dir
print(f"Current working directory: {original_dir}")
os.makedirs("Plots", exist_ok=True)

param_in_file_path         = os.path.join(run_directory, "param.in")
add_param_file_path        = os.path.join(run_directory, "additional_param.txt")
GRN_config_file_path       = os.path.join(run_directory, "config.txt")
kinetics_file_path         = os.path.join(run_directory, "kinetics.txt")
interactions_file_path     = os.path.join(run_directory, "interactions.txt")
random_before_signal_file_path = os.path.join(run_directory, "random_before_signal.txt")

somatic_coordinates_dir = run_directory
germ_coordinates_dir    = run_directory
os.makedirs(somatic_coordinates_dir, exist_ok=True)
os.makedirs(germ_coordinates_dir,    exist_ok=True)

coordinate_files = {
    'sphere': {
        'somatic': os.path.join(somatic_coordinates_dir, "Somatic_coordinates_sphere.txt"),
        'germ':    os.path.join(germ_coordinates_dir,    "Germ_coordinates_sphere.txt")
    },
    'ellipsoid': {
        'somatic': os.path.join(somatic_coordinates_dir, "Somatic_coordinates_ellipsoid.txt"),
        'germ':    os.path.join(germ_coordinates_dir,    "Germ_coordinates_ellipsoid.txt")
    },
    'cylinder': {
        'somatic': os.path.join(somatic_coordinates_dir, "Somatic_coordinates_cylinder.txt"),
        'germ':    os.path.join(germ_coordinates_dir,    "Germ_coordinates_cylinder.txt")
    }
}

quantfile_dir = os.path.join(run_directory, "Quant_Files")
if not os.path.exists(quantfile_dir):
    os.makedirs(quantfile_dir)
    print(f"Created directory: {quantfile_dir}")
else:
    print(f"Directory already exists: {quantfile_dir}")


SWEEP_MODE = "one-at-a-time" # "one-at-a-time" "grid"
python_params_choice = "python_params_baseline.txt" # "python_params_all.txt"  "python_params_baseline.txt" # based on the choice of parameter file, the simulation will run with different parameters.



Current working directory: /home/pkumar/Documents/Gitlab_INRIA/oocyte-selection-3d-abm/Codes/run/local_run
Created directory: /home/pkumar/Documents/Gitlab_INRIA/oocyte-selection-3d-abm/Codes/run/local_run/Quant_Files


In [3]:
def build_project():
    ## Now build the project 

    # just for debugging purposes, print the paths to Python and cmake
    print("Python:", sys.executable)
    print("PATH:", os.environ.get("PATH"))
    print("cmake:", shutil.which("cmake"))

    # Define paths
    project_root = os.path.abspath("../../")  # Path to the project root (where CMakeLists.txt is located)
    build_dir = os.path.join(project_root, "build")  # Path to the build directory
    executable_path = os.path.join(build_dir, "bin", "simuscale")  # Path to the executable

    # Step 1: Create the build directory
    if not os.path.exists(build_dir):
        os.makedirs(build_dir)
        print(f"Created build directory: {build_dir}")
    else:
        print(f"Build directory already exists: {build_dir}")

    # Step 2: Run cmake and make
    try:
        # Run cmake
        print("Running cmake...")
        subprocess.run(["cmake", ".."], cwd=build_dir, check=True)

        # Run make
        print("Running make...")
        subprocess.run(["make"], cwd=build_dir, check=True)

        print("Build completed successfully.")
    except subprocess.CalledProcessError as e:
        print(f"Error during build: {e}")
        exit(1)

# Call the function
build_project()


Python: /home/pkumar/simuscale-venv/bin/python
PATH: /home/pkumar/simuscale-venv/bin:/home/pkumar/bin:/home/pkumar/bin:/home/pkumar/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/usr/games:/usr/local/games:/snap/bin:/snap/bin
cmake: /usr/bin/cmake
Created build directory: /home/pkumar/Documents/Gitlab_INRIA/oocyte-selection-3d-abm/Codes/build
Running cmake...


-- The CXX compiler identification is GNU 13.3.0
-- The C compiler identification is GNU 13.3.0
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Display detected (Linux desktop) → BUILD_VIEW defaults ON
-- BUILD_VIEW = ON
-- Configuring for Linux
-- Conda not active → using system GSL
-- Found GSL: /usr/include (found version "2.7.1") 
-- GSL configured successfully
-- Found OpenGL: /usr/lib/x86_64-linux-gnu/libOpenGL.so   
-- Found GLUT: /usr/lib/x86_64-linux-gnu/libglut.so  
-- Found ZLIB: /usr/lib/x86_64-linux-gnu/libz.so (found version "1.3")  
-- Found PNG: /usr/lib/x86_64-linux-gnu/libpng.so (found version "1.6.43") 
-- Fou

Re-execute cmake to add or change plugins.


-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE  
-- Configuring done (1.0s)
-- Generating done (0.0s)
-- Build files have been written to: /home/pkumar/Documents/Gitlab_INRIA/oocyte-selection-3d-abm/Codes/build
Running make...
[  2%] Generating InterCellSignal.h
[  5%] Generating CellType.h
[  5%] Built target generated_headers
[  7%] Building CXX object src/simuscale-core/CMakeFiles/simuscale-core.dir/Alea.cpp.o
[ 10%] Building CXX object src/simuscale-core/CMakeFiles/simuscale-core.dir/Cell.cpp.o
[ 13%] Building CXX object src/simuscale-core/CMakeFiles/simuscale-core.dir/Grid.cpp.o
[ 15%] Building CXX object src/simuscale-core/CMakeFiles/simuscale-core.dir/Observable.cpp.o
[ 18%] Building CXX object src/simuscale-core/CMakeFiles/simuscale-core.dir/OutputManager.cpp.o
[ 21%] Building CXX object src/simuscale-core/CMakeFiles/simuscale-core.dir/Population.cpp.o
[ 23%] Building CXX object src/simuscale-core/CMakeFiles/simuscale-core.dir/Simulation.cpp.o
[ 26%

In [4]:
def cleanup_previous_runs():
    """Clean up files from previous simulation runs"""
    print("🧹 Cleaning up previous simulation files...")

    # Files to remove from run directory
    files_to_remove = [
        "combined_nurse_cell_activation.txt",
        "mean_summary.txt", 
        "cv_summary.txt",
        "timing_summary.txt"
    ]

    # Remove individual result files
    for file in files_to_remove:
        file_path = os.path.join(run_directory, file)
        try:
            if os.path.exists(file_path):
                os.remove(file_path)
                print(f"  ✅ Removed {file}")
            else:
                print(f"  ⏭️ Skipped {file} (doesn't exist)")
        except Exception as e:
            print(f"  ⚠️ Could not remove {file}: {e}")

    # Clean Quant_Files directory
    try:
        if os.path.exists(quantfile_dir):
            # Remove all files in Quant_Files
            for filename in os.listdir(quantfile_dir):
                file_path = os.path.join(quantfile_dir, filename)
                try:
                    if os.path.isfile(file_path):
                        os.remove(file_path)
                except Exception as e:
                    print(f"  ⚠️ Could not remove {filename}: {e}")
            print(f"  ✅ Cleaned Quant_Files directory")
        else:
            # Create directory if it doesn't exist
            os.makedirs(quantfile_dir)
            print(f"  ✅ Created Quant_Files directory")
    except Exception as e:
        print(f"  ⚠️ Error with Quant_Files directory: {e}")

    # Clean individual nurse cell activation files
    try:
        nurse_cell_pattern = os.path.join(run_directory, "nurse_cell_activation_counts_*.txt")
        files_found = glob.glob(nurse_cell_pattern)
        removed_count = 0
        
        for file_path in files_found:
            try:
                if os.path.exists(file_path):
                    os.remove(file_path)
                    removed_count += 1
            except Exception as e:
                print(f"  ⚠️ Could not remove {os.path.basename(file_path)}: {e}")
        
        print(f"  ✅ Removed {removed_count} nurse cell activation files")
    except Exception as e:
        print(f"  ⚠️ Error cleaning nurse cell files: {e}")

    # Clean worker directories (optional - removes all previous simulation subdirectories)
    try:
        cmd = f'find {run_directory} -type d -name "sim_*" -path "*/worker_*/*" -exec rm -rf {{}} +'
        subprocess.run(cmd, shell=True, check=True)
        print("  ✅ Cleaned worker simulation subdirectories")
    except subprocess.CalledProcessError as e:
        print(f"  ⚠️ Shell cleanup failed: {e}")

    print("🧹 Cleanup completed!")

cleanup_previous_runs()

🧹 Cleaning up previous simulation files...
  ⏭️ Skipped combined_nurse_cell_activation.txt (doesn't exist)
  ⏭️ Skipped mean_summary.txt (doesn't exist)
  ⏭️ Skipped cv_summary.txt (doesn't exist)
  ⏭️ Skipped timing_summary.txt (doesn't exist)
  ✅ Cleaned Quant_Files directory
  ✅ Removed 0 nurse cell activation files
  ✅ Cleaned worker simulation subdirectories
🧹 Cleanup completed!


In [5]:
def update_normalization_file(run_directory, simulationIdentifier, additional_param_file):
    ## read the additional param file and change the normalization file for visualization
    ## currently it is not being used, can be used to change the normalization file for visualization.

    # open the file
    with open(os.path.join(run_directory, additional_param_file), "r") as file:
        # take the values alpha and beta
        alpha = 0.0
        beta = 0.0
        for line in file:
            if "alpha" in line:
                alpha = float(line.split()[1])
            if "beta" in line:
                beta = float(line.split()[1])
    print(alpha, beta)

    NBD_carrying_capacity = 1.0 ##0.8 * alpha / (alpha + beta)
    print(NBD_carrying_capacity)

    # First read the file normalization.txt
    # open the file normalization.txt
    with open(os.path.join(run_directory, f"Quant_Files/normalization_{simulationIdentifier}.txt"), "r") as file:
        # read the content of the file
        content = file.read()
    # print the content of the file
    print(content)

    # open the file normalization_new.txt or create it if it does not exist
    lines = content.split('\n')
    new_lines = []
    for line in lines:
        if "GERM" in line and "GERM_NBD" not in line:
            new_lines.append("GERM 1.0 0.0")
        elif "GERM_NBD_SPHERICAL" in line:
            new_lines.append("GERM_NBD_SPHERICAL " + str(NBD_carrying_capacity) + " 0.0")
        elif "GERM_NBD_CYLINDRICAL" in line:
            new_lines.append("GERM_NBD_CYLINDRICAL " + str(NBD_carrying_capacity) + " 0.0")
        elif "GERM_NBD_ELLIPSOIDAL" in line:
            new_lines.append("GERM_NBD_ELLIPSOIDAL " + str(NBD_carrying_capacity) + " 0.0")
        else:
            new_lines.append(line)

    new_content = '\n'.join(new_lines)
    with open(os.path.join(run_directory, f"Quant_Files/normalization_new_{simulationIdentifier}.txt"), "w") as file:
        file.write(new_content)
        file.write("\n# Customized Normalization File\n")

    # open the file normalization_new.txt read, print and close it
    with open(os.path.join(run_directory, f"Quant_Files/normalization_new_{simulationIdentifier}.txt"), "r") as file:
        print(file.read())


In [6]:
def update_coordinates(file_path, new_somatic_coordinate_file, new_germ_coordinate_file):
    """
    Updates the param.in file to use specific coordinate files.
    
    Parameters:
    - file_path: Path to the param.in file
    - new_somatic_coordinate_file: Shape name for somatic coordinates
    - new_germ_coordinate_file: Shape name for germ coordinates
    """
    # Get absolute paths for coordinate files (for copying purposes)
    somatic_file = coordinate_files[new_somatic_coordinate_file]['somatic']
    germ_file = coordinate_files[new_germ_coordinate_file]['germ']
    
    # Get just the filenames for param.in references
    somatic_filename = f"Somatic_coordinates_{new_somatic_coordinate_file}.txt"
    germ_filename = f"Germ_coordinates_{new_germ_coordinate_file}.txt"
    
    # print(f"Setting somatic coordinates file: {somatic_filename}")
    # print(f"Setting germ coordinates file: {germ_filename}")
    
    with open(file_path, 'r') as file:
        lines = file.readlines()

    with open(file_path, 'w') as file:
        for line in lines:
            if 'Somatic_coordinates_' in line:
                line = re.sub(r'Somatic_coordinates_\S+', somatic_filename, line)
            if 'Germ_coordinates_' in line:
                line = re.sub(r'Germ_coordinates_\S+', germ_filename, line)
            file.write(line)

In [7]:
def update_additional_param_file(file_path, factor, PTN_threshold, nurse_cell_activation_file,
                                  alpha, beta, mass_transfer_enabled, reference_nuclei_factor):
    with open(file_path, 'r') as f:
        lines = f.readlines()
    with open(file_path, 'w') as f:
        for line in lines:
            if 'nbdFactor' in line:
                line = re.sub(r'nbdFactor \S+', f'nbdFactor {factor}', line)
            if 'PTN_threshold' in line:
                line = re.sub(r'PTN_threshold \S+', f'PTN_threshold {PTN_threshold}', line)
            if 'nurse_cell_activation_file' in line:
                line = re.sub(r'nurse_cell_activation_file \S+',
                              f'nurse_cell_activation_file {nurse_cell_activation_file}', line)
            if 'alpha' in line and 'alpha' == line.strip().split()[0]:
                line = re.sub(r'alpha \S+', f'alpha {alpha}', line)
            if 'beta' in line and 'beta' == line.strip().split()[0]:
                line = re.sub(r'beta \S+', f'beta {beta}', line)
            if 'mass_transfer_enabled' in line and 'mass_transfer_enabled' == line.strip().split()[0]:
                line = re.sub(r'mass_transfer_enabled \S+',
                              f'mass_transfer_enabled {int(mass_transfer_enabled)}', line)
            if 'reference_nuclei_factor' in line and 'reference_nuclei_factor' == line.strip().split()[0]:
                line = re.sub(r'reference_nuclei_factor \S+',
                              f'reference_nuclei_factor {reference_nuclei_factor}', line)
            f.write(line)

In [8]:
def update_GRN_config_file(file_path, External_Signal_Coeff, Phi):
    with open(file_path, 'r') as f:
        lines = f.readlines()
    with open(file_path, 'w') as f:
        for line in lines:
            if 'External_Signal_Coeff' in line:
                line = re.sub(r'(External_Signal_Coeff\s*:\s*\S+).*',
                              f'External_Signal_Coeff: {External_Signal_Coeff}', line)
            if 'Basal_activity_Phi' in line:
                line = re.sub(r'(Basal_activity_Phi\s*:\s*\S+).*',
                              f'Basal_activity_Phi: {Phi}', line)
            f.write(line)
               

In [9]:
# =============================================================================
# Helper — update_kinetics_file 
# Updates kinetics.txt with per-simulation values of d0, d1, a0, a1, a2, d0_2
# Assumes each parameter appears as:   param_name   value   (space-separated)
# =============================================================================
def update_kinetics_file(file_path, d0, d1, a0, a1, a2, d0_2):
    kinetic_params = {
        'd0':   d0,
        'd1':   d1,
        'a0':   a0,
        'a1':   a1,
        'a2':   a2,
        'd0_2': d0_2,
    }
    with open(file_path, 'r') as f:
        lines = f.readlines()
    with open(file_path, 'w') as f:
        for line in lines:
            stripped = line.strip()
            if not stripped or stripped.startswith('#'):
                f.write(line)
                continue
            parts = stripped.split()
            if len(parts) >= 2 and parts[0] in kinetic_params:
                param_name = parts[0]
                new_val    = kinetic_params[param_name]
                # Replace only the numeric value, preserve rest of line
                line = re.sub(
                    rf'^(\s*{re.escape(param_name)}\s+)\S+',
                    lambda m: f'{m.group(1)}{new_val}',
                    line
                )
            f.write(line)


In [10]:
# Helper — update_interactions_file  
# Updates interactions.txt entries for a 2x2 matrix:
# interaction_1_1, interaction_1_2, interaction_2_1, interaction_2_2
# =============================================================================
def update_interactions_file(file_path, i11, i12, i21, i22):
    interaction_params = {
        'interaction_1_1': i11,
        'interaction_1_2': i12,
        'interaction_2_1': i21,
        'interaction_2_2': i22,
    }
    with open(file_path, 'r') as f:
        lines = f.readlines()
    with open(file_path, 'w') as f:
        for line in lines:
            stripped = line.strip()
            if not stripped or stripped.startswith('#'):
                f.write(line)
                continue
            parts = stripped.split()
            if len(parts) >= 2 and parts[0] in interaction_params:
                param_name = parts[0]
                new_val    = interaction_params[param_name]
                line = re.sub(
                    rf'^(\s*{re.escape(param_name)}\s+)\S+',
                    lambda m: f'{m.group(1)}{new_val}',
                    line
                )
            f.write(line)

In [11]:
# Function to read the value where the first column (TimeStep) is 240.0
def read_last_nurse_cell_activation(file_path):
    try:
        with open(file_path, "r") as file:
            lines = file.readlines()
            for line in lines[1:]:  # Skip header
                parts = line.strip().split("\t")
                if len(parts) >= 4 and float(parts[0]) == 240.0:  # Check for 4 columns
                    nurse_count = float(parts[1])
                    total_germ_count = int(parts[2])
                    pro_oocyte_count = int(parts[3])  
                    return nurse_count, total_germ_count, pro_oocyte_count
            return None
    except FileNotFoundError:
        #print(f"File {file_path} not found.")
        return None



In [12]:
# Function to append results to the summary file
def append_to_summary_file(output_file, PTN_threshold, External_Signal_Coeff, alpha, beta, nurse_value):
    with open(output_file, "a") as file:
        file.write(f"{PTN_threshold:.2f}\t{External_Signal_Coeff:.2f}\t{alpha:.4f}\t{beta:.4f}\t{nurse_value:.2f}\n")

In [13]:
# Helper — read_txt_params
# =============================================================================
LINSPACE_PARAMS = {
    'PTN_threshold', 'External_Signal_Coeff', 'alpha', 'beta',
    'size_factor', 'reference_nuclei_factor',
    'd0', 'd1', 'a0', 'a1', 'a2', 'd0_2', 'Phi',
    'interaction_1_1', 'interaction_1_2', 'interaction_2_1', 'interaction_2_2',
}

def read_txt_params(txt_file):
    params = {}
    with open(txt_file, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            if '=' in line:
                key, value = line.split('=', 1)
                key   = key.strip()
                value = value.strip()

                if key in ['shapes', 'sizes']:
                    params[key] = [v.strip() for v in value.split(',')]
                elif key in LINSPACE_PARAMS:
                    parts = [v.strip() for v in value.split(',')]
                    min_val, max_val, points = float(parts[0]), float(parts[1]), int(float(parts[2]))
                    params[key] = np.linspace(min_val, max_val, points)
                elif key in ['total_runs', 'mass_transfer_enabled', 'save_trajectory_files']:
                    params[key] = int(value)
                else:
                    params[key] = value
    return params


In [14]:
def combine_nurse_cell_activation_data(input_file, output_file, simulation_identifier, reset_file=False):
    """
    Reads nurse cell activation data from individual simulation runs and combines them into a single file.
    
    Parameters:
    - input_file: Path to the nurse_cell_activation_counts.txt file
    - output_file: Path to the combined data file
    - simulation_identifier: Identifier for the current simulation to use as column header
    - reset_file: If True, clears the output file before writing (used at the start of a new simulation set)
    """
    try:
        # Read the current simulation data
        data = pd.read_csv(input_file, sep="\t")

        # Reset file if requested
        if reset_file:
            with open(output_file, "w") as f:
                pass  # Clears the file
            # print(f"[INFO] Output file reset: {output_file}")

        # If output file doesn't exist or was just reset, write header and data
        if not os.path.exists(output_file) or os.path.getsize(output_file) == 0:
            with open(output_file, "w") as f:
                f.write(f"TimeStep\t{simulation_identifier}\n")
                for index, row in data.iterrows():
                    f.write(f"{row['TimeStep']}\t{row['NurseCellCount']}\n")
            # print(f"[INFO] Created new combined file: {output_file}")
        
        else:
            # File exists with data, so append new simulation column
            existing_data = pd.read_csv(output_file, sep="\t")

            # Time point mismatch warning
            if len(existing_data) != len(data):
                print(f"[WARNING] Time points in {simulation_identifier} don't match existing data.")

            # Add new data column
            existing_data[simulation_identifier] = data['NurseCellCount'].values[:len(existing_data)]

            # Save updated file
            existing_data.to_csv(output_file, sep="\t", index=False)
            # print(f"[INFO] Appended data for {simulation_identifier} to {output_file}")
    
    except Exception as e:
        print(f"[ERROR] Failed to combine data for {simulation_identifier}: {e}")


In [15]:
# Create persistent working directories for each core
def initialize_working_dirs(num_cores):
    working_dirs = []
    for i in range(num_cores):
        work_dir = os.path.join(run_directory, f"worker_{i}")
        os.makedirs(work_dir, exist_ok=True)

        for shape in shapes:
            somatic_file = coordinate_files[shape]['somatic']
            germ_file    = coordinate_files[shape]['germ']
            if os.path.exists(somatic_file) and os.path.exists(germ_file):
                shutil.copy(somatic_file, os.path.join(work_dir, os.path.basename(somatic_file)))
                shutil.copy(germ_file,    os.path.join(work_dir, os.path.basename(germ_file)))

        for src_path, dest_name in [
            (kinetics_file_path,             "kinetics.txt"),
            (interactions_file_path,         "interactions.txt"),
            (random_before_signal_file_path, "random_before_signal.txt"),
            (param_in_file_path,             "param.in"),
            (add_param_file_path,            "additional_param.txt"),
            (GRN_config_file_path,           "config.txt"),
        ]:
            if os.path.exists(src_path):
                shutil.copy(src_path, os.path.join(work_dir, dest_name))

        working_dirs.append(work_dir)
    return working_dirs

In [16]:
def save_timing_summary(output_file, timing_data):
    with open(output_file, "w") as f:
        f.write("=== Simulation Timing Summary ===\n")
        f.write(f"Generated at: {time.strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        for key, value in timing_data.items():
            if isinstance(value, list) and value:
                f.write(f"{key}:\n")
                f.write(f"  Average: {np.mean(value):.3f}s\n")
                f.write(f"  Min:     {np.min(value):.3f}s\n")
                f.write(f"  Max:     {np.max(value):.3f}s\n")
                f.write(f"  Total:   {np.sum(value):.3f}s\n\n")
            else:
                f.write(f"{key}: {value}\n")



In [17]:

# params added to signature: d0, d1, a0, a1, a2, d0_2, Phi,
#                                 i11, i12, i21, i22
# =============================================================================
def run_single_simulation(
    simulation_run, PTN_threshold, External_Signal_Coeff, alpha, beta,
    size_factor, reference_nuclei_factor,
    d0, d1, a0, a1, a2, d0_2, Phi,
    i11, i12, i21, i22,
    work_dir, add_param_file_path, GRN_config_file_path,
    nurse_cell_activation_file_template, shape,
    simulationIdentifier, mass_transfer_flag, save_trajectory_files
):
    original_cwd   = os.getcwd()
    sim_start_time = time.time()

    try:
        # --- Setup ---
        setup_start  = time.time()
        sim_work_dir = os.path.join(work_dir, f"sim_{simulation_run}")
        os.makedirs(sim_work_dir, exist_ok=True)

        required_files = [
            "param.in", "additional_param.txt", "config.txt",
            "kinetics.txt", "interactions.txt", "random_before_signal.txt",
            f"Somatic_coordinates_{shape}.txt", f"Germ_coordinates_{shape}.txt"
        ]
        for filename in required_files:
            src = os.path.join(work_dir, filename)
            if os.path.exists(src):
                shutil.copy(src, os.path.join(sim_work_dir, filename))

        nurse_cell_activation_file = os.path.abspath(
            nurse_cell_activation_file_template.format(simulationIdentifier=simulationIdentifier)
        )
        os.makedirs(os.path.dirname(nurse_cell_activation_file), exist_ok=True)
        setup_time = time.time() - setup_start

        # --- Parameter Updates ---
        param_start      = time.time()
        local_add_param  = os.path.join(sim_work_dir, "additional_param.txt")
        local_GRN_config = os.path.join(sim_work_dir, "config.txt")
        local_param_in   = os.path.join(sim_work_dir, "param.in")
        local_kinetics   = os.path.join(sim_work_dir, "kinetics.txt")
        local_interactions = os.path.join(sim_work_dir, "interactions.txt")

        update_additional_param_file(local_add_param, size_factor, PTN_threshold,
                                     nurse_cell_activation_file, alpha, beta,
                                     mass_transfer_flag, reference_nuclei_factor)
        update_GRN_config_file(local_GRN_config, External_Signal_Coeff, Phi)
        update_coordinates(local_param_in, shape, shape)
        update_kinetics_file(local_kinetics, d0, d1, a0, a1, a2, d0_2)        
        update_interactions_file(local_interactions, i11, i12, i21, i22)              
        param_time = time.time() - param_start

        # --- C++ Execution ---
        os.chdir(sim_work_dir)
        project_root   = os.path.abspath(os.path.join(os.getcwd(), "../../../.."))
        cpp_executable = os.path.join(project_root, "build/bin/simuscale")

        if not os.path.exists(cpp_executable):
            raise FileNotFoundError(f"C++ executable not found: {cpp_executable}")

        cpp_start = time.time()
        try:
            subprocess.run([cpp_executable], capture_output=True, text=True,
                           check=True, timeout=300)
        except subprocess.CalledProcessError as e:
            print(f"[{simulationIdentifier}] FAILED with exit code {e.returncode}")
            return None, None, None, None, None, None, None
        except subprocess.TimeoutExpired:
            print(f"[{simulationIdentifier}] Timed out after 300 seconds")
            return None, None, None, None, None, None, None
        cpp_time = time.time() - cpp_start

        # --- Output Processing ---
        output_start = time.time()
        if save_trajectory_files:
            missing_files = []
            if not os.path.exists("normalization.txt"):
                missing_files.append("normalization.txt")
            if not os.path.exists("trajectory.txt"):
                missing_files.append("trajectory.txt")
            if missing_files:
                print(f"[ERROR] Missing files {missing_files} after {simulationIdentifier}")
                for mf in missing_files:
                    with open(mf, 'w') as f:
                        f.write(f"# Empty {mf} file (original missing)\n")
            os.makedirs(quantfile_dir, exist_ok=True)
            shutil.copy("trajectory.txt",
                        os.path.join(quantfile_dir, f"trajectory_{simulationIdentifier}.txt"))
            shutil.copy("normalization.txt",
                        os.path.join(quantfile_dir, f"normalization_{simulationIdentifier}.txt"))

        os.chdir(original_cwd)
        last_nurse_germ_values = read_last_nurse_cell_activation(nurse_cell_activation_file)

        try:
            shutil.rmtree(sim_work_dir)
        except Exception as e:
            print(f"[WARNING] Could not remove {sim_work_dir}: {e}")

        output_time = time.time() - output_start
        total_time  = time.time() - sim_start_time

        return (simulationIdentifier, last_nurse_germ_values, nurse_cell_activation_file,
                total_time, setup_time, param_time, cpp_time, output_time)

    except Exception as e:
        print(f"[ERROR] Simulation {simulation_run} failed: {str(e)}")
        traceback.print_exc()
        os.chdir(original_cwd)
        return (None, None, None, 0, 0, 0, 0, 0)


In [18]:
# =============================================================================
# Load Parameters from python_params.txt
# =============================================================================
file_choice = python_params_choice 
param_file  = os.path.join(run_directory, python_params_choice) 
sim_params  = read_txt_params(param_file)

shapes                       = sim_params['shapes']
sizes                        = sim_params['sizes']
PTN_threshold_values         = sim_params['PTN_threshold']
External_Signal_Coeff_values = sim_params['External_Signal_Coeff']
alpha_values                 = sim_params['alpha']
beta_values                  = sim_params['beta']
size_factor_values           = sim_params['size_factor']
reference_nuclei_factor_values = sim_params['reference_nuclei_factor']
total_runs                   = sim_params['total_runs']
mass_transfer_enabled        = sim_params['mass_transfer_enabled']
save_trajectory_files        = sim_params['save_trajectory_files']

# Kinetic params
d0_values   = sim_params['d0']
d1_values   = sim_params['d1']
a0_values   = sim_params['a0']
a1_values   = sim_params['a1']
a2_values   = sim_params['a2']
d0_2_values = sim_params['d0_2']
Phi_values  = sim_params['Phi']

# Interaction matrix params
i11_values  = sim_params['interaction_1_1']
i12_values  = sim_params['interaction_1_2']
i21_values  = sim_params['interaction_2_1']
i22_values  = sim_params['interaction_2_2']

In [19]:
# =============================================================================
# Main Loop
# =============================================================================
if __name__ == "__main__":
    mp.set_start_method("fork", force=True)
    num_cpus = mp.cpu_count() - 1
    print(f"Running simulations in parallel using {num_cpus} cores")

    working_dirs = initialize_working_dirs(num_cpus)

    # ---- Output file paths ----
    combined_data_file    = os.path.join(run_directory, "combined_nurse_cell_activation.txt")
    mean_summary_file     = os.path.join(run_directory, "mean_summary.txt")
    cv_summary_file       = os.path.join(run_directory, "cv_summary.txt")
    germ_summary_file     = os.path.join(run_directory, "germ_cell_summary.txt")
    all_combinations_file = os.path.join(run_directory, "all_combinations_results.txt")

    with open(mean_summary_file, "w") as f:
        f.write("PTN_threshold\tExternal_Signal_Coeff\talpha\tbeta\tMean_Last_NurseCellActivation\n")
    with open(cv_summary_file, "w") as f:
        f.write("PTN_threshold\tExternal_Signal_Coeff\talpha\tbeta\tCV_Last_NurseCellActivation\n")

    with open(germ_summary_file, "w") as f:
        headers = [
            "[P_n]_0", "zeta", "alpha", "beta",
            "lambda", "V_nucleus",
            "d0_o", "d1", "a0", "a1", "a2", "d0_n", "Phi",
            "theta11", "theta12", "theta21", "theta22",
            "Mean_Nurse", "CV_Nurse",
            "Mean_Germ", "CV_Germ",
            "Mean_Oocyte", "CV_Oocyte",
            "N/G_Ratio",
        ]
        f.write('\t'.join(h.ljust(10) for h in headers) + '\n')
    
    with open(all_combinations_file, "w") as f:
        f.write(
            "SimulationID\tShape\tSize\t"
            "P_n\tzeta\talpha\tbeta\t"
            "lambda\tV_nucleus\t"
            "d0_o\td1\ta0\ta1\ta2\td0_n\tPhi\t"
            "theta_11\ttheta_12\ttheta_21\ttheta_22\t"
            "Run\tNurseCells\tTotalGermCells\tOocytes\n"
        )

    # ------------------------------------------------------------------
    # Build one-at-a-time sweep sets or all combinations (grid) based on SWEEP_MODE
    # Each entry in sweep_sets is a full dict of ALL parameter values.
    # We start from the baseline (index [0] of each linspace) and then
    # vary one parameter at a time across its range.
    # ------------------------------------------------------------------
    ALL_SWEEP_PARAMS = {
        'PTN_threshold':          PTN_threshold_values,
        'External_Signal_Coeff':  External_Signal_Coeff_values,
        'alpha':                  alpha_values,
        'beta':                   beta_values,
        'size_factor':            size_factor_values,
        'reference_nuclei_factor': reference_nuclei_factor_values,
        'd0':                     d0_values,
        'd1':                     d1_values,
        'a0':                     a0_values,
        'a1':                     a1_values,
        'a2':                     a2_values,
        'd0_2':                   d0_2_values,
        'Phi':                    Phi_values,
        'interaction_1_1':        i11_values,
        'interaction_1_2':        i12_values,
        'interaction_2_1':        i21_values,
        'interaction_2_2':        i22_values,
    }

    baseline = {k: float(v[len(v) // 2]) for k, v in ALL_SWEEP_PARAMS.items()}

    if SWEEP_MODE == "grid":
        keys = list(ALL_SWEEP_PARAMS.keys())
        unique_sets = [dict(zip(keys, (float(x) for x in vals)))
                   for vals in itertools.product(*[ALL_SWEEP_PARAMS[k] for k in keys])]
    else:
        print("Performing one-at-a-time parameter sweep...")
        sweep_sets = [baseline.copy()]
        for param_name, param_values in ALL_SWEEP_PARAMS.items():
            if len(param_values) > 1:
                for val in param_values:
                    combo = baseline.copy()
                    combo[param_name] = float(val)
                    sweep_sets.append(combo)
        unique_sets = []
        seen = set()
        for s in sweep_sets:
            key = tuple(round(s[k], 8) for k in sorted(ALL_SWEEP_PARAMS.keys()))
            if key not in seen:
                seen.add(key)
                unique_sets.append(s)
        print(f"One-at-a-time sweep: {len(unique_sets)} unique parameter combinations")


    # ------------------------------------------------------------------
    # Build full parameter combination list
    # ------------------------------------------------------------------
    all_parameter_combinations = []
    simulation_id = 0

    for shape in shapes:
        for combo in unique_sets:
            for simulation_run in range(total_runs):
                all_parameter_combinations.append({
                    'simulation_id':          simulation_id,
                    'simulation_run':         simulation_run,
                    'shape':                  shape,
                    'size_label':             's',   # kept for identifier; size comes from combo
                    **combo,
                })
                simulation_id += 1

    total_simulations = len(all_parameter_combinations)
    print(f"Total simulations: {total_simulations}")
    print(f"Processing approximately {total_simulations // num_cpus} tasks per core")

    # ------------------------------------------------------------------
    # Batch parallel execution
    # ------------------------------------------------------------------
    job_start_time = time.time()
    all_results    = []

    for batch_start in range(0, total_simulations, num_cpus):
        batch_end = min(batch_start + num_cpus, total_simulations)
        print(f"Processing batch {batch_start // num_cpus + 1}: "
              f"simulations {batch_start + 1}-{batch_end}")

        batch_tasks = []
        for task_idx in range(batch_start, batch_end):
            p = all_parameter_combinations[task_idx]

            # Build a compact simulation identifier
            simulationIdentifier = (
                f"{p['shape']}_{p['size_label']}"
                f"_PTN_{p['PTN_threshold']:.2f}"
                f"_ESC_{p['External_Signal_Coeff']:.2f}"
                f"_A_{p['alpha']:.4f}_B_{p['beta']:.4f}"
                f"_SF_{p['size_factor']:.3f}_RNF_{p['reference_nuclei_factor']:.3f}"
                f"_d0_{p['d0']:.3f}_d1_{p['d1']:.4f}"
                f"_a0_{p['a0']:.3f}_a1_{p['a1']:.2f}_a2_{p['a2']:.3f}"
                f"_d02_{p['d0_2']:.3f}_Phi_{p['Phi']:.3f}"
                f"_i11_{p['interaction_1_1']:.2f}_i12_{p['interaction_1_2']:.3f}"
                f"_i21_{p['interaction_2_1']:.3f}_i22_{p['interaction_2_2']:.2f}"
                f"_Run_{p['simulation_run'] + 1}"
            )

            worker_index = task_idx - batch_start
            nurse_cell_activation_file_template = os.path.join(
                run_directory,
                "nurse_cell_activation_counts_{simulationIdentifier}.txt"
            )

            batch_tasks.append((
                p['simulation_run'],
                p['PTN_threshold'],
                p['External_Signal_Coeff'],
                p['alpha'],
                p['beta'],
                p['size_factor'],
                p['reference_nuclei_factor'],
                p['d0'], p['d1'], p['a0'], p['a1'], p['a2'], p['d0_2'], p['Phi'],
                p['interaction_1_1'], p['interaction_1_2'],
                p['interaction_2_1'], p['interaction_2_2'],
                working_dirs[worker_index],
                os.path.join(working_dirs[worker_index], "additional_param.txt"),
                os.path.join(working_dirs[worker_index], "config.txt"),
                nurse_cell_activation_file_template,
                p['shape'],
                simulationIdentifier,
                mass_transfer_enabled,
                save_trajectory_files,
            ))

        with mp.Pool(num_cpus) as pool:
            batch_results = pool.starmap(run_single_simulation, batch_tasks)

        all_results.extend(batch_results)
        print(f"Completed batch {batch_start // num_cpus + 1}/"
              f"{(total_simulations - 1) // num_cpus + 1}")

    # ------------------------------------------------------------------
    # Process all results
    # ------------------------------------------------------------------
    print(f"Processing {len(all_results)} results...")

    param_groups        = {}
    all_simulation_data = {}
    total_times = []; setup_times = []; param_times = []; cpp_times = []; output_times = []

    for result in all_results:
        if result[0] is None:
            continue

        (simulationIdentifier, last_nurse_germ_values, nurse_cell_file,
         total_time, setup_time, param_time, cpp_time, output_time) = result

        total_times.append(total_time);  setup_times.append(setup_time)
        param_times.append(param_time);  cpp_times.append(cpp_time)
        output_times.append(output_time)

        if last_nurse_germ_values is not None and last_nurse_germ_values[0] is not None:
            nurse_count, total_germ_count, pro_oocyte_count = last_nurse_germ_values


            # 
            id_to_params = {
                (
                    f"{c['shape']}_{c['size_label']}"
                    f"_PTN_{c['PTN_threshold']:.2f}"
                    f"_ESC_{c['External_Signal_Coeff']:.2f}"
                    f"_A_{c['alpha']:.4f}_B_{c['beta']:.4f}"
                    f"_SF_{c['size_factor']:.3f}_RNF_{c['reference_nuclei_factor']:.3f}"
                    f"_d0_{c['d0']:.3f}_d1_{c['d1']:.4f}"
                    f"_a0_{c['a0']:.3f}_a1_{c['a1']:.2f}_a2_{c['a2']:.3f}"
                    f"_d02_{c['d0_2']:.3f}_Phi_{c['Phi']:.3f}"
                    f"_i11_{c['interaction_1_1']:.2f}_i12_{c['interaction_1_2']:.3f}"
                    f"_i21_{c['interaction_2_1']:.3f}_i22_{c['interaction_2_2']:.2f}"
                    f"_Run_{c['simulation_run'] + 1}"
                ): c
                for c in all_parameter_combinations
            }

                        # ✅ Exact lookup
            matched = id_to_params.get(simulationIdentifier, None)

            if matched:
                param_key = tuple(round(matched[k], 8) for k in sorted(ALL_SWEEP_PARAMS.keys()))
                param_key = (matched['shape'], matched['size_label']) + param_key

                if param_key not in param_groups:
                    param_groups[param_key] = {
                        'params':            matched,
                        'nurse_counts':      [],
                        'germ_counts':       [],
                        'pro_oocyte_counts': [],
                        'nurse_cell_files':  [],
                    }
                param_groups[param_key]['nurse_counts'].append(nurse_count)
                param_groups[param_key]['germ_counts'].append(total_germ_count)
                param_groups[param_key]['pro_oocyte_counts'].append(pro_oocyte_count)
                param_groups[param_key]['nurse_cell_files'].append(nurse_cell_file)

                # ---- Write every run to all-in-one file ----
                m       = matched
                run_num = simulationIdentifier.split("_Run_")[-1]
                with open(all_combinations_file, "a") as f:
                    f.write(
                        f"{simulationIdentifier}\t{m['shape']}\t{m['size_label']}\t"
                        f"{m['PTN_threshold']}\t{m['External_Signal_Coeff']}\t"
                        f"{m['alpha']}\t{m['beta']}\t"
                        f"{m['size_factor']}\t{m['reference_nuclei_factor']}\t"
                        f"{m['d0']}\t{m['d1']}\t{m['a0']}\t{m['a1']}\t{m['a2']}\t"
                        f"{m['d0_2']}\t{m['Phi']}\t"
                        f"{m['interaction_1_1']}\t{m['interaction_1_2']}\t"
                        f"{m['interaction_2_1']}\t{m['interaction_2_2']}\t"
                        f"{run_num}\t{nurse_count:.4f}\t{total_germ_count}\t{pro_oocyte_count}\n"
                    )
            # Collect time-series data
            try:
                if os.path.exists(nurse_cell_file):
                    sim_data = pd.read_csv(
                        nurse_cell_file, sep="\t",
                        usecols=[0, 1, 2, 3],
                        names=["TimeStep", "NurseCellCount", "TotalGermCount", "ProOocyteCount"],
                        header=0, on_bad_lines='skip'
                    )
                    all_simulation_data[simulationIdentifier] = sim_data
            except Exception as e:
                print(f"[WARNING] Could not read data for {simulationIdentifier}: {e}")

    # ------------------------------------------------------------------
    # Build combined time-series file
    # ------------------------------------------------------------------
    print(f"Creating combined data file with {len(all_simulation_data)} simulations...")

    if all_simulation_data:
        data_dict = {}
        first_sim = list(all_simulation_data.values())[0]
        data_dict['TimeStep'] = first_sim['TimeStep'].reset_index(drop=True)

        for sim_id in sorted(all_simulation_data.keys()):
            sim_data   = all_simulation_data[sim_id]
            min_length = min(len(data_dict['TimeStep']), len(sim_data))
            for col_suffix, col_name in [("_Nurse", "NurseCellCount"),
                                          ("_Germ",  "TotalGermCount"),
                                          ("_ProOocyte", "ProOocyteCount")]:
                col = sim_data[col_name].iloc[:min_length].reset_index(drop=True)
                if len(col) < len(data_dict['TimeStep']):
                    padding = [np.nan] * (len(data_dict['TimeStep']) - len(col))
                    col = pd.concat([col, pd.Series(padding)], ignore_index=True)
                data_dict[f"{sim_id}{col_suffix}"] = col

        combined_df = pd.DataFrame(data_dict)
        combined_df.to_csv(combined_data_file, sep="\t", index=False)
        print(f"Combined data saved with {len(combined_df.columns) - 1} columns")

    # ------------------------------------------------------------------
    # Write aggregated summary files
    # ------------------------------------------------------------------
    for param_key, data in param_groups.items():
        m             = data['params']
        nurse_list    = data['nurse_counts']
        germ_list     = data['germ_counts']
        oocyte_list   = data['pro_oocyte_counts']
        nurse_files   = data['nurse_cell_files']

        if nurse_list:
            mean_nurse   = np.mean(nurse_list)
            cv_nurse     = np.std(nurse_list)  / mean_nurse   if mean_nurse   != 0 else 0
            mean_germ    = np.mean(germ_list)   if germ_list   else 0
            cv_germ      = np.std(germ_list)    / mean_germ    if mean_germ    != 0 else 0
            mean_oocyte  = np.mean(oocyte_list) if oocyte_list else 0
            cv_oocyte    = np.std(oocyte_list)  / mean_oocyte  if mean_oocyte  != 0 else 0
            ratio        = mean_nurse / mean_germ if mean_germ != 0 else 0

            PTN = m['PTN_threshold']; ESC = m['External_Signal_Coeff']
            A   = m['alpha'];         B   = m['beta']

            append_to_summary_file(mean_summary_file, PTN, ESC, A, B, mean_nurse)
            append_to_summary_file(cv_summary_file,   PTN, ESC, A, B, cv_nurse)
            with open(germ_summary_file, "a") as f:
                vals = [
                    f"{m['PTN_threshold']:.2f}",
                    f"{m['External_Signal_Coeff']:.2f}",
                    f"{m['alpha']:.4f}",
                    f"{m['beta']:.4f}",
                    f"{m['size_factor']:.3f}",
                    f"{m['reference_nuclei_factor']:.3f}",
                    f"{m['d0']:.3f}",
                    f"{m['d1']:.3f}",
                    f"{m['a0']:.3f}",
                    f"{m['a1']:.3f}",
                    f"{m['a2']:.3f}",
                    f"{m['d0_2']:.3f}",
                    f"{m['Phi']:.3f}",
                    f"{m['interaction_1_1']:.3f}",   
                    f"{m['interaction_1_2']:.3f}",   
                    f"{m['interaction_2_1']:.3f}",  
                    f"{m['interaction_2_2']:.3f}",   
                    f"{mean_nurse:.3f}",
                    f"{cv_nurse:.3f}",
                    f"{mean_germ:.3f}",
                    f"{cv_germ:.3f}",
                    f"{mean_oocyte:.3f}",
                    f"{cv_oocyte:.3f}",
                    f"{ratio:.3f}",
                ]
                f.write('\t'.join(v.ljust(10) for v in vals) + '\n')



            print(f"  PTN={PTN:.2f} ESC={ESC:.2f} A={A:.4f} B={B:.4f} "
                  f"SF={m['size_factor']:.3f} RNF={m['reference_nuclei_factor']:.3f} "
                  f"a0={m['a0']:.3f} → "
                  f"Nurse={mean_nurse:.2f} Germ={mean_germ:.2f} "
                  f"ProOocyte={mean_oocyte:.2f} ({len(nurse_list)} runs)")

        for nf in nurse_files:
            if os.path.exists(nf):
                os.remove(nf)

    # ------------------------------------------------------------------
    # Timing & cleanup
    # ------------------------------------------------------------------
    total_job_time = time.time() - job_start_time

    if total_times:
        print(f"\nAverage simulation times:")
        print(f"  Total:             {np.mean(total_times):.3f}s")
        print(f"  Setup:             {np.mean(setup_times):.3f}s")
        print(f"  Parameters:        {np.mean(param_times):.3f}s")
        print(f"  C++ execution:     {np.mean(cpp_times):.3f}s")
        print(f"  Output processing: {np.mean(output_times):.3f}s")

    print(f"\nAll simulations complete.")
    print(f"Total job time: {total_job_time:.2f}s ({total_job_time / 60:.2f} min)")
    print(f"Job ended at {time.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"\n📄 All individual results → {all_combinations_file}")

    timing_file = os.path.join(run_directory, "timing_summary.txt")
    save_timing_summary(timing_file, {
        "Total Job Time":          total_job_time,
        "Number of CPU Cores":     num_cpus,
        "Total Simulations":       len([r for r in all_results if r[0] is not None]),
        "Setup Times":             setup_times,
        "Parameter Update Times":  param_times,
        "C++ Execution Times":     cpp_times,
        "Output Processing Times": output_times,
    })
    print(f"Timing summary saved to: {timing_file}")

    try:
        cmd = f'find {run_directory} -maxdepth 1 -type d -name "worker_*" -exec rm -rf {{}} +'
        subprocess.run(cmd, shell=True, check=True)
        print("✅ Removed all worker directories after simulation completion")
    except subprocess.CalledProcessError as e:
        print(f"⚠️  Worker directory cleanup failed: {e}")

Running simulations in parallel using 11 cores
Performing one-at-a-time parameter sweep...
One-at-a-time sweep: 1 unique parameter combinations
Total simulations: 35
Processing approximately 3 tasks per core
Processing batch 1: simulations 1-11
Completed batch 1/4
Processing batch 2: simulations 12-22
Completed batch 2/4
Processing batch 3: simulations 23-33
Completed batch 3/4
Processing batch 4: simulations 34-35
Completed batch 4/4
Processing 35 results...
Creating combined data file with 35 simulations...
Combined data saved with 105 columns
  PTN=0.40 ESC=50.00 A=0.0300 B=0.2000 SF=1.000 RNF=0.200 a0=1.000 → Nurse=0.91 Germ=7.94 ProOocyte=7.03 (35 runs)

Average simulation times:
  Total:             6.894s
  Setup:             0.001s
  Parameters:        0.003s
  C++ execution:     6.883s
  Output processing: 0.006s

All simulations complete.
Total job time: 27.48s (0.46 min)
Job ended at 2026-11-19 14:17:22

📄 All individual results → /home/pkumar/Documents/Gitlab_INRIA/oocyte-s